# 19 · Create the train/test objects

Splits the model input object into the four files the combinatorial models
read.

| Output | Contents |
|---|---|
| `par_save_filename_trainsingles` | single-module cells plus the control training half |
| `par_save_filename_doubles` | cells perturbed in two modules |
| `par_save_filename_doubles_samegroup` | cells with both knockouts in one module |
| `par_save_filename_testcontrol` | the held-out control half |

**Reads** `par_save_filename_12`.

The control split is positional: the first `par_n_control_train` control cells
train, the next `par_n_control_test` test. It follows the row order notebook 18
produces and is not randomised.

## Setup

In [ ]:
from libraries import *
from parameters import *
from pathlib import Path

os.chdir(projectDir)
import anndata as ad

Path(par_dataset_dir).mkdir(parents=True, exist_ok=True)

In [ ]:
adata = sc.read(par_save_filename_12)
print(f"input: {adata.shape[0]} cells x {adata.shape[1]} genes")

MODULE_COLS = [c for c in adata.obs.columns
               if c.startswith("K_") and c != "K_CONTROL"] + ["K_CONTROL"]
print(f"module columns: {MODULE_COLS}")

n_labels = adata.obs[MODULE_COLS].sum(axis=1)
adata = adata[n_labels <= 2].copy()
print(f"cells with at most two module labels: {adata.shape[0]}")
print(adata.obs[MODULE_COLS].sum().to_string())

## Split the controls

In [ ]:
control = adata[adata.obs["K_CONTROL"] == 1].copy()
print(f"control cells: {control.shape[0]}")

control_train = control[0:par_n_control_train].copy()
control_test = control[par_n_control_train:par_n_control_train + par_n_control_test].copy()
print(f"  train: {control_train.shape[0]}")
print(f"  test : {control_test.shape[0]}")

if control.shape[0] < par_n_control_train + par_n_control_test:
    print("\nWARNING: fewer control cells than the split expects; the test half is short.")

control_test.write(par_save_filename_testcontrol)
print(f"\nwritten: {par_save_filename_testcontrol}")

## Doubles

In [ ]:
doubles = adata[adata.obs[MODULE_COLS].sum(axis=1) == 2].copy()
doubles.write(par_save_filename_doubles)
print(f"doubles: {doubles.shape[0]} cells -> {par_save_filename_doubles}")

doubles_same = adata[(adata.obs["Doubles"] == 1) & (adata.obs["DoubleSameGroup"] == 1)].copy()
doubles_same.write(par_save_filename_doubles_samegroup)
print(f"same-module doubles: {doubles_same.shape[0]} cells -> {par_save_filename_doubles_samegroup}")

print("\nsame-module doubles per module:")
print(doubles_same.obs[MODULE_COLS].sum().to_string())

## Train singles

Single-module perturbed cells, restricted to those carrying exactly one
knockout that appears in the guide-module table, concatenated with the control
training half.

In [ ]:
singles = adata[adata.obs[MODULE_COLS].sum(axis=1) == 1].copy()
singles_pert = singles[singles.obs["K_CONTROL"] == 0].copy()
print(f"single-module perturbed cells: {singles_pert.shape[0]}")

guide_modules = pd.read_csv(par_guideModules_file, index_col=0)
module_genes = pd.unique("GENE_" + guide_modules.GuideName + "_")
module_genes = [g for g in module_genes if g in singles_pert.obs.columns]

exactly_one = singles_pert.obs[module_genes].sum(axis=1) == 1
singles_pert = singles_pert[exactly_one].copy()
print(f"  carrying exactly one module knockout: {singles_pert.shape[0]}")

train_singles = ad.concat([singles_pert, control_train], join="inner",
                          label="batch", keys=["0", "1"], index_unique="-")
train_singles.uns["feature_barcode_names_filtered_GENES"] = singles_pert.uns.get(
    "feature_barcode_names_filtered_GENES", [])

train_singles.write(par_save_filename_trainsingles)
print(f"\ntrain singles: {train_singles.shape[0]} cells -> {par_save_filename_trainsingles}")
print(train_singles.obs[MODULE_COLS].sum().to_string())

## Check

In [ ]:
for path in (par_save_filename_trainsingles, par_save_filename_doubles,
             par_save_filename_doubles_samegroup, par_save_filename_testcontrol):
    p = Path(path)
    status = "ok" if p.exists() else "MISSING"
    size = f"{p.stat().st_size / 1e6:.0f} MB" if p.exists() else "-"
    print(f"  {status:<8} {size:>10}  {p.name}")